# Sample Creation

## Setup
The first part of this notebook defines all jubilee labware 
 -out of the 6 deck beds, 5 are in use: trashcan, tiprack, stock solutions, samples plate and the    aliquoted samples. Each one also has all of the necesary manual offsets for calibration puposes

In [ ]:
# this step reloads machine in order to prevent any error
%load_ext autoreload
%autoreload 2

In [3]:
# import statements
import requests

# ----------- Science Jubilee -------------
from science_jubilee import Machine as Jub
from science_jubilee.tools import Pipette
import time

import numpy as np
import pandas as pd
import os

In [ ]:
# defines the jubilee to call
jubilee = Jub.Machine(address='192.168.1.2', simulated = False) 

In [ ]:
# load deck
deck = jubilee.load_deck('lab_automation_deck.json')

In [ ]:
# this loads the well plate for the destination for samples (sample_holder_name, position)
samples = jubilee.load_labware('septavialrev1_44_holder_2000ul.json', 2)
# this line of code defines the boundary conditions of the well plate requires 3 points
upper_left = (19.9,183.2)
upper_right = (132.4, 183.4)
lower_right = (132.5, 113.0)
samples.manual_offset([upper_left,upper_right,lower_right])
samples.load_manualOffset()

In [ ]:
# plan on having a samples_trash for an alloquat plate to keep a constant concentration
samples_trash = jubilee.load_labware('septavialrev1_44_holder_2000ul.json',5) #used for the aloquates
upper_left = (19.3,279.8)
upper_right = (132.0, 279.8)
lower_right = (132.0, 209.8)
samples_trash.manual_offset([upper_left,upper_right,lower_right])
samples_trash.load_manualOffset()

In [ ]:
# this is same as lines above just for stock solution
stocks = jubilee.load_labware('20mlscintillation_12_wellplate_18000ul.json', 3)
upper_left = (167.4,176.8)
upper_right = (258.0, 176.8)
lower_right = (258.0, 119.8)
stocks.manual_offset([upper_left,upper_right,lower_right])
stocks.load_manualOffset()

In [ ]:
# same as above just for pipette tip rack
tiprack = jubilee.load_labware('opentrons_96_tiprack_300ul.json', 0)
upper_left = (27.0,82.7)
upper_right = (125.6,82.7)
lower_right = (125.8,19.4)
tiprack.manual_offset([upper_left,upper_right,lower_right])
tiprack.load_manualOffset()

In [ ]:
# defines position of trash can for tips
trash = jubilee.load_labware('agilent_1_reservoir_290ml.json', 1)

In [ ]:
deck.safe_z

In [ ]:
# this code block defines all of our stocks
water_stock = stocks[0].bottom(+5) 
blue1_stock = stocks[1].bottom(+5) #stock A: 0.10 M
blue2_stock = stocks[2].bottom(+5) #         0.15 M
blue3_stock = stocks[3].bottom(+5) #         0.20 M
red1_stock = stocks[4].bottom(+5)  #stock B: 0.10 M
red2_stock = stocks[5].bottom(+5)  #         0.15 M
red3_stock = stocks[6].bottom(+5)  #         0.20 M
# include however many we need, remember to include however many stocks there are
# this list will be a reference lise that will be used to return the index value of which stock to choose 
reference_stocks_blue = [0, 0.1, 0.15, 0.2]
reference_stocks_red = [0,0,0,0, 0.1, 0.15, 0.2]

## Load Tools

In [ ]:
P300 = Pipette.Pipette.from_config(3,'Pipette','P300_config.json')
jubilee.load_tool(P300)

P300.add_tiprack(tiprack)
P300.trash = trash[0]

## Experiment Functions
 4 Main functions are written: transfer stock, transfer mix, transfer water and mixer
     transfer stock simply moves some volume from some stock to the final wellplate
     transfer mix includes the aliquot and mixing step that occurs at the final step of experimentprocess
     transfer water is just for water
     mixer just mixes and aliquots incase only that function is needed

In [ ]:
def transfer_stock(v, stock, target_sample):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - stock (int) -position of the desired stock
          - sample (int) -position of the current sample
    Output - void, purpose of function    
    '''
    P300.transfer(v, source_well=stocks[stock].bottom(+2) ,
                  destination_well = samples[target_sample].bottom(+5), 
                  blowout=True, 
                  new_tip='once' )

In [ ]:
def transfer_mix(v, stock, sample, transfer_index):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - stock (int) index location of stock
          - sample (int) sample index location
 
    Output - void, purpose of function    
    '''
    # get the required material
    P300.pickup_tip()
    P300.aspirate(v, stocks[stock].bottom(+2))
    P300.dispense(v, samples[sample].bottom(+4))
    P300.blowout()

    # manual mixing step
    P300.mix(300,5)

    # move to alliquot
    P300.aspirate(300, samples[sample].bottom(+3))
    P300.dispense(300, samples_trash[transfer_index].bottom(+3))
    P300.blowout()
    P300.drop_tip()

In [ ]:
def transfer_water(v,sample):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - sample (int)
    Output - void, purpose of function    
    '''
    # source well is always the same
    P300.transfer(v, source_well=stocks[0].bottom(+3),
                  destination_well = samples[sample].bottom(+5), 
                  blowout=True, 
                  new_tip='once' )

In [ ]:
def mixer(sample_location_index, sample_location_trash_index): # hold off on this function for now
    '''
    this function will mix a chosen sample, it is done this way in order to reuse the same tip for mixing and not using the stock solution tip
    Input - sample_location(int) is an index location of which sample to mix
    Output - none
    '''
    P300.pickup_tip()
    P300.aspirate(1, samples[sample_location_index].bottom(+5))
    P300.dispense(1, samples[sample_location_index].bottom(+5))
    
    P300.mix(300,5)
    
    P300.aspirate(300, samples[sample_location_index].bottom(+3))
    P300.dispense(300, samples_trash[sample_location_trash_index].bottom(+5))
    P300.drop_tip()

# Data for experiment #
the 2 main loops at the end account for the blocks that run the program above, the rest is all preparational information for the proper values to be input

In [ ]:
os.getcwd()

In [ ]:
data = pd.read_csv('./data/for_igi.2.0.csv')

In [23]:
def data_extractor(sample_file_name):
    '''
    This function creates a sample list that reads data files and creates them into a list form from csv
    input - string - enter the name of the file to read from
    output - dictionary - dictionary of lists that will be used to run sample creator functions
    '''
    data = pd.read_csv(sample_file_name)
    vol_a = data['vol_A']
    vol_b = data['vol_B']
    vol_water = data['vol_water']
    stock_a = data['stock_A']
    stock_b = data['stock_B']
    a_index_list = []
    b_index_list = []
    vol_a = vol_a * 1000/3.3 # to scale to the P300 pipette
    vol_b = vol_b * 1000/3.3
    vol_water = vol_water * 1000/3.3

    for i in range(len(stock_a)):
        temp = reference_stocks_blue.index(stock_a[i])
        a_index_list.append(temp)

    for i in range(len(stock_b)):
        temp = reference_stocks_red.index(stock_b[i])
        b_index_list.append(temp)

    d = {'vol_a':vol_a, 'vol_b':vol_b, 'vol_water':vol_water, 
         'a_index_list':a_index_list, 'b_index_list':b_index_list
    }
    
    return d

In [6]:
# this is first time step, all the others can be done by increasing the index
# the dt1 is a list of [vol_a, vol_b, vol_h20, stock_a, stock_b]
#dt1 = data.loc[0]
#data.head()
reference_stocks_blue = [0, 0.1, 0.2, 0.3]
reference_stocks_red = [0, 0, 0, 0, 0.10, 0.15, 0.20] # first zeros 

In [16]:
samples_list = [] #all samples into list
sampleA = data_extractor('./data/for_igi.2.0.csv')
#sampleB = data_extractor('')
samples_list.append(sampleA) #etc you get the idea

## Run Experiment Code Lines

In [ ]:
jubilee.pickup_tool(P300) # line to pickup tool

In [ ]:
# for loop for a single sample. first add water, then cobalt, then phosphate
'''for i in range(len(sampleA['vol_a'])):
    if sampleA['vol_water'][i] > 0:
        transfer_water(sampleA['vol_water'][i],0)
    if sampleA['vol_a'][i] > 0:
        transfer_stock(sampleA['vol_a'][i],sampleA['a_index_list'][i],0)
    if sampleA['vol_b'][i] > 0:
        transfer_mix(sampleA['vol_b'][i],sampleA['b_index_list'][i],0,i)
    if sampleA['vol_b'][i] == 0:
        mixer(i)'''

In [ ]:
# when there are multiple sample files here is to run them all at same time
'''smp_tr_indx_ct = 0
for i in range(len(sampleA['vol_a'])): #indexes by sample volume
    for j in range(len(samples_list)): #indexes by sample number
        if samples_list[j]['vol_water'][i] > 0:
            transfer_water(samples_list[j]'vol_water'[i],j)
        if samples_list[j]['vol_a'][i] > 0:
            transfer_stock(samples_list[j]'vol_a'[i],j)
        if samples_list[j]['vol_b'][i] > 0:
            transfer_mix(samples_list[j]'vol_b'[i],j,smp_tr_indx_ct)
        if samples_list[j]['vol_b'][i] == 0:
            mixer(j, smp_tr_indx_ct)
        smp_tr_indx_ct += 1 #add to the aliquot count'''

In [ ]:
def begin(samp):
    '''
    initializes a sample by creating the first state with 500 uL of 1M phosphate
    :param samp: location of the sample
    :return:
    '''
    # transfer 250 uL of water and 2M phosphate
    transfer_water(250,samp)
    transfer_stock(250,2,samp)

In [ ]:
def rxn_quencher(k):
     P300.transfer(300, source_well=stocks[0].bottom(+3),
                  destination_well = samples_trash[k].bottom(+4),
                  blowout=True,
                  new_tip='once' )

In [ ]:
def sun(samp,po, k):
    '''
    Creates the history for a complete sample
    :param samp:
    :return:
    '''
    df = pd.read_csv(samp)
    volumes_a = df['vol_co']
    volumes_b = df['vol_b']
    volumes_water = df['vol_w']
    for i in range(len(volumes_a)):
        if i != 0:
            if volumes_a[i] > 0:
                transfer_stock(volumes_a[i],1,po)
            if volumes_water[i] > 0:
                transfer_water(volumes_water[i],po)
            if volumes_b[i] > 0:
                transfer_mix(volumes_b[i],2,po, k)
                rxn_quencher(k)
                k = k+1

In [7]:
kl = 0
for j in range(2):
    begin(j)
    if j == 0:
        sun('Sample_w_History_1',0, kl)
    if j == 1:
        sun('Sample_w_History_2',1, kl)
    kl=kl+5

SyntaxError: invalid syntax (1199354809.py, line 4)